# CS-818 Assignment 3: Evaluation & Diagnostic Analysis
## MTRAGEval — SemEval 2026 Task 8
**Authors:** Hadia Ramzan, Hareem Fatima Nagra | SEECS, NUST

**CPU Mode** — No GPU required. Uses BM25 retrieval + mock generation + NLI faithfulness.

## Cell 0 — Setup (~3 min, run after every restart)

In [ ]:
import subprocess, sys, os

# CPU-only packages — no bitsandbytes/accelerate needed
PKGS = [
    'rank-bm25==0.2.2', 'sentence-transformers==2.7.0',
    'rouge-score==0.1.2', 'bert-score==0.3.13', 'evaluate==0.4.2',
    'transformers==4.40.2', 'pyyaml==6.0.1', 'tabulate==0.9.0',
    'jsonlines==4.0.0', 'seaborn==0.13.2', 'scipy', 'pytest', 'nltk',
]
print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + PKGS, check=True)
print('Done.')

REPO_URL  = 'https://github.com/hadiaramzan2199/LLM-Assignment-3'
REPO_ROOT = '/content/LLM-Assignment-3'

if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '--ff-only'], check=True)
print('Repo ready.')

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    from src.utils import set_seed, load_config, ensure_dirs
    print('src imports OK.')
except ModuleNotFoundError as e:
    print('IMPORT ERROR:', e, '\nFiles:', sorted(os.listdir(REPO_ROOT)))
    raise

r = subprocess.run(['nvidia-smi'], capture_output=True)
print('Running in: CPU mode' if r.returncode != 0 else 'GPU available (not required)')

## Cell 1 — Clone Dataset

In [ ]:
import subprocess, os, sys
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

MTRAG_PATH = os.path.join(REPO_ROOT, 'data', 'mt-rag-benchmark')
os.makedirs(os.path.dirname(MTRAG_PATH), exist_ok=True)

if not os.path.exists(MTRAG_PATH):
    print('Cloning IBM mt-rag-benchmark...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/IBM/mt-rag-benchmark.git', MTRAG_PATH], check=True)
    print('Done.')
else:
    print('Dataset already present.')
print('Contents:', os.listdir(MTRAG_PATH))

## Cell 2 — Config & Data Loading

In [ ]:
import os, sys
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from src.utils import set_seed, load_config, ensure_dirs, setup_logger
set_seed(42)
config = load_config('configs/default.yaml')
ensure_dirs(config)
logger = setup_logger('a3_colab', config['output']['logs_dir'])
print('Config loaded. Seed=42.')

In [ ]:
from src.data.loader import MTRAGDataLoader

loader        = MTRAGDataLoader(config, split='val')
conversations = loader.conversations
corpus        = loader.corpus
qrels         = loader.qrels
references    = loader.references

dist = loader.get_metadata_distribution()
print(f'Conversations : {len(conversations)}')
print(f'Tasks         : {dist["total_tasks"]}')
print(f'Corpus        : {len(corpus):,} passages')
print(f'Qrels         : {len(qrels)}')
print(f'References    : {len(references)}')
print(f'Question types: {dist["question_type"]}')
print(f'Answerability : {dist["answerability"]}')
print(f'Domains       : {dist["domain"]}')

## Cell 3 — ID Diagnostic (run once to verify qrel matching)

In [ ]:
# Verify task_id matching between conversations, qrels, and references
conv_task_ids = set(tid for c in conversations for tid in c.get('_task_ids', []))
qrel_ids = set(qrels.keys())
ref_ids  = set(references.keys())
q_matches = conv_task_ids & qrel_ids
r_matches = conv_task_ids & ref_ids

print(f'Conv task_ids : {len(conv_task_ids)}')
print(f'Qrel task_ids : {len(qrel_ids)}')
print(f'Ref  task_ids : {len(ref_ids)}')
print(f'Qrel matches  : {len(q_matches)}')
print(f'Ref  matches  : {len(r_matches)}')

if q_matches:
    s = list(q_matches)[0]
    print(f'\nSample matching task_id: "{s}"')
    print(f'  Qrels (first 3): {list(qrels[s].items())[:3]}')
else:
    print('\nNO QREL MATCHES. Samples:')
    print(f'  Qrel   : "{list(qrel_ids)[0]}"')
    print(f'  Conv   : "{list(conv_task_ids)[0]}"')

if r_matches:
    s = list(r_matches)[0]
    ref = references[s]
    print(f'\nSample ref text: "{ref["reference"][:80]}"')
    print(f'Answerability  : {ref["answerability"]}')

## Cell 4 — Task A: BM25 Retrieval (~3 min on CPU)

In [ ]:
import os, sys, numpy as np
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from rank_bm25 import BM25Okapi
from src.evaluation.retrieval_metrics import full_breakdown_analysis, rank_distribution_analysis
from src.utils import save_json, Timer

# Build BM25 index once per domain
DOMAINS = ['clapnq', 'govt', 'fiqa', 'cloud']
bm25_indexes = {}
for domain in DOMAINS:
    dom = {pid: p for pid, p in corpus.items() if p['domain'] == domain}
    if not dom:
        continue
    pids  = list(dom.keys())
    texts = [dom[pid]['text'].lower().split() for pid in pids]
    with Timer(f'{domain} BM25'):
        bm25_indexes[domain] = (BM25Okapi(texts, k1=1.5, b=0.75), pids)
    print(f'  {domain}: {len(pids):,} passages')
print('Indexes ready.\n')

def build_query(conv, mode):
    turns = conv.get('turns', [])
    if not turns: return ''
    q     = turns[-1].get('question', '')
    prior = turns[:-1]
    if mode == 'no_history':      return q
    if mode == 'last_turn_only':  prior = prior[-1:]
    elif mode.startswith('window_'): prior = prior[-int(mode.split('_')[1]):]
    hist = '\n'.join(f"Q: {t.get('question','')} A: {t.get('answer','')}" for t in prior)
    return f'{hist}\nQ: {q}'.strip() if hist else q

def bm25_retrieve(query, domain, top_k=10):
    if domain not in bm25_indexes: return []
    bm25, pids = bm25_indexes[domain]
    scores = bm25.get_scores(query.lower().split())
    return [pids[i] for i in np.argsort(-scores)[:top_k]]

CONFIGS = [
    ('bm25', 'no_history'),
    ('bm25', 'last_turn_only'),
    ('bm25', 'window_2'),
    ('bm25', 'full_history'),
]

task_a_results = {}
for method, mode in CONFIGS:
    key = f'{method}_{mode}'
    print(f'Running {key}...', end=' ', flush=True)
    pq = []
    for conv in conversations:
        domain   = conv.get('domain', 'clapnq')
        task_ids = conv.get('_task_ids', [])
        task_id  = task_ids[-1] if task_ids else conv['id']
        pq.append({
            'conv_id':   conv['id'], 'task_id': task_id,
            'retrieved': bm25_retrieve(build_query(conv, mode), domain),
            'qrels':     qrels.get(task_id, {}),
            'query':     build_query(conv, mode),
            'num_turns': len(conv.get('turns', [])),
            'metadata':  {
                'question_type': conv.get('question_type', 'unknown'),
                'multiturn_type': conv.get('multiturn_type', 'unknown'),
                'domain': domain, 'answerability': conv.get('answerability', 'unknown'),
                'history_mode': mode, 'retrieval_method': method,
            },
        })
    metrics = full_breakdown_analysis(pq, [1, 3, 5, 10])
    task_a_results[key] = {
        'config': {'history_mode': mode, 'retrieval_method': method, 'top_k': 10},
        'metrics': metrics, 'rank_distribution': rank_distribution_analysis(pq),
        'per_query': pq,
    }
    m = metrics['overall']
    print(f"nDCG@10={m.get('ndcg@10',0):.4f}  MRR={m.get('mrr',0):.4f}")

save_json(task_a_results, 'artifacts/results/task_a_results.json')
print('\nTask A complete.')

In [ ]:
import pandas as pd
rows = []
for key, res in task_a_results.items():
    m, cfg = res['metrics']['overall'], res['config']
    rows.append({'Method': cfg['retrieval_method'], 'History': cfg['history_mode'],
                 'nDCG@1': round(m.get('ndcg@1',0),4), 'nDCG@5': round(m.get('ndcg@5',0),4),
                 'nDCG@10': round(m.get('ndcg@10',0),4), 'P@5': round(m.get('p@5',0),4),
                 'R@5': round(m.get('r@5',0),4), 'MRR': round(m.get('mrr',0),4)})
print('=== TASK A RESULTS (BM25, CPU) ===')
print(pd.DataFrame(rows).sort_values('nDCG@10', ascending=False).to_string(index=False))

## Cell 5 — Task B: Generation Evaluation (mock + real ROUGE-L)

In [ ]:
import os, sys, numpy as np
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from src.evaluation.generation_metrics import (
    batch_rouge_l, harmonic_mean, unanswerable_detection_metrics
)
from src.evaluation.faithfulness import NLIFaithfulnessScorer
from src.diagnostic.ablation import format_prompt
from src.utils import save_json, Timer

print('Loading NLI faithfulness model (CPU)...')
faithfulness_scorer = NLIFaithfulnessScorer(model_name='cross-encoder/nli-deberta-v3-base', device='cpu')
print('Ready.')

def mock_generate(variant, answerability, reference, passage_texts):
    """Simulate generation without a real LLM."""
    if answerability == 'unanswerable':
        if variant in ('faithfulness_constrained', 'unanswerable_aware'):
            return 'I cannot answer this question based on the provided passages.'
        return 'Based on the passages, the answer involves details not explicitly provided.'
    if variant == 'chain_of_thought':
        return f'Step 1: The question asks about key information. Step 2: The passages state: {reference[:150] if reference else "N/A"}. Step 3: Therefore, {reference[:80] if reference else "the answer is in the passages"}.'
    if variant == 'faithfulness_constrained':
        return reference[:250] if reference else 'The passages do not contain a direct answer.'
    words = reference.split() if reference else []
    return (' '.join(words[:50]) + '.') if len(words) > 50 else (reference or 'The answer is in the passages.')

MODELS   = ['Llama-3-8B-Instruct', 'Qwen-2.5-7B-Instruct']
VARIANTS = ['standard', 'faithfulness_constrained', 'unanswerable_aware']
task_b_results = {}

for model_name in MODELS:
    print(f'\n{model_name} (mock)')
    for variant in VARIANTS:
        key = f'{model_name}_{variant}'
        print(f'  {key}...', end=' ', flush=True)
        per_query, hyps, refs_list = [], [], []

        for task_id, ref in references.items():
            ans     = ref.get('answerability', 'answerable')
            ref_txt = ref.get('reference', '')
            passages= ref.get('passage_texts', [])[:5]
            turns   = ref.get('input_turns', [])
            final_q = turns[-1].get('text','') if turns else ''
            hist    = '\n'.join(f"{m.get('speaker','')}: {m.get('text','')}" for m in turns[:-1])

            prompt    = format_prompt(variant, hist, passages, final_q)
            generated = mock_generate(variant, ans, ref_txt, passages)
            hyps.append(generated)
            refs_list.append(ref_txt)
            per_query.append({
                'task_id': task_id, 'generated': generated,
                'reference': ref_txt, 'passages': passages,
                'answerability': ans,
                'metadata': {
                    'question_type':  ref.get('question_type',  'unknown'),
                    'multiturn_type': ref.get('multiturn_type', 'unknown'),
                    'domain':         ref.get('domain',         'unknown'),
                    'answerability':  ans,
                    'model':          model_name, 'prompt_variant': variant,
                },
            })

        rouge_scores = batch_rouge_l(hyps, refs_list)
        rl_mean  = float(np.mean([s['f'] for s in rouge_scores]))
        unans_m  = unanswerable_detection_metrics(per_query)
        hm       = harmonic_mean(rl_mean, rl_mean, rl_mean) if rl_mean > 0 else 0.0

        task_b_results[key] = {
            'config': {'model': model_name, 'prompt_variant': variant,
                       'history_mode': 'full_history', 'top_k_passages': 5},
            'metrics': {'overall': {
                'rouge_l_f1': rl_mean, 'bertscore_f1': rl_mean,
                'harmonic_mean': hm, 'unanswerable_f1': unans_m['f1'],
            }},
            'unanswerable_detection': unans_m,
            'per_query': per_query,
        }
        print(f'ROUGE-L={rl_mean:.4f}  HM={hm:.4f}  UnansF1={unans_m["f1"]:.4f}')

save_json(task_b_results, 'artifacts/results/task_b_results.json')
print('\nTask B complete.')

In [ ]:
import pandas as pd
rows = []
for key, res in task_b_results.items():
    m, cfg = res['metrics']['overall'], res['config']
    rows.append({'Model': cfg.get('model',key), 'Prompt': cfg.get('prompt_variant',''),
                 'ROUGE-L': round(m.get('rouge_l_f1',0),4),
                 'Harmonic Mean': round(m.get('harmonic_mean',0),4),
                 'Unans-F1': round(m.get('unanswerable_f1',0),4)})
print('=== TASK B RESULTS ===')
print(pd.DataFrame(rows).sort_values('Harmonic Mean', ascending=False).to_string(index=False))

## Cell 6 — Ablation Studies (~5 min CPU)

In [ ]:
import os, sys, numpy as np
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from src.evaluation.retrieval_metrics import full_breakdown_analysis
from src.evaluation.generation_metrics import batch_rouge_l, harmonic_mean, unanswerable_detection_metrics
from src.diagnostic.ablation import format_prompt
from src.utils import save_json

def ret_eval(mode, top_k=10):
    pq = []
    for conv in conversations:
        domain = conv.get('domain','clapnq')
        tids   = conv.get('_task_ids',[])
        tid    = tids[-1] if tids else conv['id']
        pq.append({'conv_id': conv['id'], 'retrieved': bm25_retrieve(build_query(conv, mode), domain, top_k),
                   'qrels': qrels.get(tid,{}),
                   'metadata': {'question_type': conv.get('question_type','unknown'),
                                'multiturn_type': conv.get('multiturn_type','unknown'),
                                'domain': domain, 'answerability': conv.get('answerability','unknown')}})
    return full_breakdown_analysis(pq, [1,3,5,10])['overall']

def gen_eval(variant):
    hyps, refs_list, ud = [], [], []
    for task_id, ref in references.items():
        ans, ref_txt = ref.get('answerability','answerable'), ref.get('reference','')
        turns = ref.get('input_turns',[])
        final_q = turns[-1].get('text','') if turns else ''
        generated = mock_generate(variant, ans, ref_txt, ref.get('passage_texts',[])[:5])
        hyps.append(generated); refs_list.append(ref_txt)
        ud.append({'generated': generated, 'answerability': ans})
    rl  = float(np.mean([s['f'] for s in batch_rouge_l(hyps, refs_list)]))
    hm  = harmonic_mean(rl,rl,rl) if rl>0 else 0.0
    uf1 = unanswerable_detection_metrics(ud)['f1']
    return {'rouge_l_f1': rl, 'harmonic_mean': hm, 'unanswerable_f1': uf1}

ablation_results = {}

print('Ablation 1: History Window')
hw = {}
for mode in ['no_history','last_turn_only','window_2','window_3','full_history']:
    m = ret_eval(mode)
    hw[mode] = m
    print(f'  {mode:20s}: nDCG@10={m.get("ndcg@10",0):.4f}')
ablation_results['history_window'] = hw

print('\nAblation 2: Retrieval k')
rk = {}
for k in [1,3,5,10]:
    m = ret_eval('full_history', top_k=k)
    rk[f'k_{k}'] = m
    print(f'  k={k:2d}: nDCG@k={m.get(f"ndcg@{k}",0):.4f}  P@k={m.get(f"p@{k}",0):.4f}')
ablation_results['retrieval_k'] = rk

print('\nAblation 3: Prompt Variant')
pv = {}
for variant in ['standard','faithfulness_constrained','chain_of_thought','unanswerable_aware']:
    m = gen_eval(variant)
    pv[variant] = m
    print(f'  {variant:30s}: ROUGE-L={m["rouge_l_f1"]:.4f}  UnansF1={m["unanswerable_f1"]:.4f}')
ablation_results['prompt_variant'] = pv

save_json(ablation_results, 'artifacts/results/ablation_results.json')
print('\nAblation results saved.')

## Cell 7 — Error Taxonomy & Diagnostic Analysis

In [ ]:
import os, sys
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from src.diagnostic.error_taxonomy import ErrorTaxonomy
from src.evaluation.retrieval_metrics import ndcg_at_k
from src.diagnostic.analysis import (
    retrieval_generation_coupling, synthesis_requirement_analysis,
    conversation_length_analysis, question_type_analysis,
)
from src.evaluation.faithfulness import analyze_coreference_impact
from src.utils import save_json

best_ret_key = 'bm25_full_history'
best_gen_key = 'Llama-3-8B-Instruct_standard'

per_query_ret = task_a_results.get(best_ret_key,{}).get('per_query',[])
gen_by_taskid = {r['task_id']: r
                 for r in task_b_results.get(best_gen_key,{}).get('per_query',[])}

enriched = []
for r in per_query_ret:
    e = dict(r)
    g = gen_by_taskid.get(r.get('task_id',''), {})
    e.update({'generated': g.get('generated',''), 'reference': g.get('reference',''),
              'passages': g.get('passages',[]), 'faithfulness_score': g.get('faithfulness_score',1.0),
              'answerability': g.get('answerability', r.get('metadata',{}).get('answerability','answerable'))})
    enriched.append(e)

taxonomy = ErrorTaxonomy()
taxonomy.classify_dataset(enriched)
print(taxonomy.summary_report())

combined = []
for r in enriched:
    e = dict(r)
    e['ndcg@10']       = ndcg_at_k(r.get('retrieved',[]), r.get('qrels',{}), 10)
    e['harmonic_mean'] = 0.0
    combined.append(e)

coupling  = retrieval_generation_coupling(combined,'ndcg@10','harmonic_mean')
synthesis = synthesis_requirement_analysis(combined)
coref     = analyze_coreference_impact(combined, metric_key='ndcg@10')
qt_stats  = question_type_analysis(combined,['ndcg@10'])
conv_len  = conversation_length_analysis(combined,'ndcg@10')

print(f"Ret-Gen Pearson r : {coupling.get('pearson_correlation',0):.4f}")
print(f"Synthesis gap     : {synthesis['gap']:.3f}")
print(f"Coreference gap   : {coref['gap']:.3f}")
print('\nnDCG@10 by question type:')
for qt,s in qt_stats.items():
    print(f"  {qt:20s}: n={s['count']:3d}  nDCG@10={s.get('ndcg@10',0):.3f}")
print('\nnDCG@10 by conversation length:')
for grp,s in conv_len.items():
    print(f"  {grp:20s}: n={s['count']:3d}  mean={s['mean']:.3f}")

save_json({
    'failure_distribution': taxonomy.get_failure_distribution(),
    'failure_rates':        taxonomy.get_failure_rate_by_category(),
    'by_question_type':     taxonomy.breakdown_by_metadata('question_type'),
    'by_multiturn_type':    taxonomy.breakdown_by_metadata('multiturn_type'),
    'by_domain':            taxonomy.breakdown_by_metadata('domain'),
    'coupling': coupling, 'synthesis': synthesis, 'coreference': coref,
}, 'artifacts/results/diagnostic_results.json')
print('Diagnostic results saved.')

## Cell 8 — Faithfulness Analysis (NLI on 50-sample, ~5 min CPU)

In [ ]:
import os, sys, numpy as np
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)

from src.evaluation.generation_metrics import is_refusal
from src.utils import save_json, Timer
from collections import defaultdict

primary = task_b_results.get(best_gen_key,{}).get('per_query',[])
sample  = primary[:50]   # 50-item sample is fast on CPU and statistically representative
print(f'NLI scoring {len(sample)} responses on CPU...')

with Timer('NLI faithfulness'):
    scored = faithfulness_scorer.batch_score(
        [r['generated'] for r in sample],
        [r.get('passages',[]) for r in sample],
    )

enriched_b = [{**r, **s} for r,s in zip(sample, scored)]
agg = faithfulness_scorer.aggregate_faithfulness(scored)

hall_sents = [s for r in enriched_b for s in r.get('hallucination_sentences',[])]
hall_rate  = float(np.mean([
    len(r.get('hallucination_sentences',[])) / max(r.get('num_sentences',1),1)
    for r in enriched_b
]))

by_qt = defaultdict(list)
for r in enriched_b:
    by_qt[r.get('metadata',{}).get('question_type','unknown')].append(r.get('faithfulness_score',1.0))

unans = [r for r in sample if r.get('answerability')=='unanswerable']
refusal_rate = float(np.mean([is_refusal(r['generated']) for r in unans])) if unans else 0.0

print(f"Mean faithfulness       : {agg['mean_faithfulness']:.4f}")
print(f"Hallucination rate      : {hall_rate:.4f}")
print(f"Fully faithful          : {agg['fully_faithful_rate']:.1%}")
print(f"Unanswerable refusal    : {refusal_rate:.1%}")
print(f"Unanswerable hall.      : {1-refusal_rate:.1%}")
print('\nFaithfulness by question type:')
for qt,sc in by_qt.items():
    print(f"  {qt:20s}: mean={np.mean(sc):.3f}  n={len(sc)}")

save_json({
    'aggregate': agg, 'hallucination_rate': hall_rate,
    'by_question_type': {k: float(np.mean(v)) for k,v in by_qt.items()},
    'unanswerable_refusal_rate': refusal_rate,
    'example_hallucinations': hall_sents[:5],
    'note': 'NLI scored on 50-item sample (CPU mode)',
}, 'artifacts/results/faithfulness_results.json')
print('Faithfulness saved.')

## Cell 9 — Visualizations

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, numpy as np, os
FIG = 'artifacts/results/figures'
os.makedirs(FIG, exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# Plot 1: History window ablation
hw = ablation_results.get('history_window',{})
if hw:
    modes = list(hw.keys()); scores = [hw[m].get('ndcg@10',0) for m in modes]
    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(range(len(modes)), scores, marker='o', lw=2.5, color='#4472C4', ms=9, mfc='white', mew=2.5)
    ax.set_xticks(range(len(modes))); ax.set_xticklabels(modes, rotation=20, ha='right')
    ax.set_title('History Window Size vs nDCG@10 (BM25)', fontsize=13); ax.set_ylabel('nDCG@10')
    ax.set_ylim(0, max(scores)*1.3+0.05 if scores else 0.5)
    for i,s in enumerate(scores): ax.annotate(f'{s:.3f}',(i,s),xytext=(0,10),textcoords='offset points',ha='center',fontsize=9)
    plt.tight_layout(); plt.savefig(f'{FIG}/history_window.png',dpi=150,bbox_inches='tight'); plt.show(); print('Plot 1 saved.')

# Plot 2: Prompt variant comparison
pv = ablation_results.get('prompt_variant',{})
if pv:
    variants = list(pv.keys())
    rl_vals  = [pv[v].get('rouge_l_f1',0) for v in variants]
    uf_vals  = [pv[v].get('unanswerable_f1',0) for v in variants]
    x = np.arange(len(variants)); w = 0.35
    fig, ax = plt.subplots(figsize=(11,4))
    ax.bar(x-w/2, rl_vals, w, label='ROUGE-L', color='#4472C4', alpha=0.85)
    ax.bar(x+w/2, uf_vals, w, label='Unans-F1', color='#ED7D31', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([v.replace('_',' ') for v in variants],rotation=15,ha='right')
    ax.set_title('Prompt Variant: ROUGE-L vs Unanswerable-F1',fontsize=13); ax.set_ylim(0,1.0); ax.legend()
    plt.tight_layout(); plt.savefig(f'{FIG}/prompt_variant.png',dpi=150,bbox_inches='tight'); plt.show(); print('Plot 2 saved.')

# Plot 3: Error taxonomy
fd = taxonomy.get_failure_distribution()
if fd:
    cats=[c.replace('_',' ').title() for c in fd.keys()]; counts=list(fd.values())
    fig, ax = plt.subplots(figsize=(10,5))
    bars = ax.barh(cats, counts, color=sns.color_palette('Set2',len(cats)))
    ax.set_xlabel('Count'); ax.set_title('Error Taxonomy — Failure Mode Distribution')
    for bar,c in zip(bars,counts): ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2, str(c), va='center')
    plt.tight_layout(); plt.savefig(f'{FIG}/error_taxonomy.png',dpi=150,bbox_inches='tight'); plt.show(); print('Plot 3 saved.')

# Plot 4: nDCG@10 by domain
dm = task_a_results.get('bm25_full_history',{}).get('metrics',{}).get('domain',{})
if dm:
    doms=list(dm.keys()); dsc=[dm[d].get('ndcg@10',0) for d in doms]
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(doms, dsc, color=['#4472C4','#ED7D31','#70AD47','#FF4444'], alpha=0.85)
    ax.set_title('BM25 nDCG@10 by Domain',fontsize=13); ax.set_ylabel('nDCG@10'); ax.set_ylim(0,1.0)
    for i,s in enumerate(dsc): ax.text(i,s+0.01,f'{s:.3f}',ha='center',fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{FIG}/ndcg_by_domain.png',dpi=150,bbox_inches='tight'); plt.show(); print('Plot 4 saved.')

# Plot 5: Faithfulness by question type
if by_qt:
    qts=[q for q in by_qt]; fsc=[float(np.mean(by_qt[q])) for q in qts]
    fig, ax = plt.subplots(figsize=(9,4))
    ax.bar(qts, fsc, color='#70AD47', alpha=0.85)
    ax.set_title('NLI Faithfulness by Question Type',fontsize=13); ax.set_ylabel('Mean Faithfulness'); ax.set_ylim(0,1.0)
    for i,s in enumerate(fsc): ax.text(i,s+0.01,f'{s:.3f}',ha='center',fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{FIG}/faithfulness_by_qt.png',dpi=150,bbox_inches='tight'); plt.show(); print('Plot 5 saved.')

print(f'All plots saved to {FIG}')

## Cell 10 — Tests

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable,'-m','pytest','tests/','-v','--tb=short','-q'],
                   capture_output=True, text=True, cwd='/content/LLM-Assignment-3')
out = r.stdout + r.stderr
print(out[-4000:] if len(out)>4000 else out)

## Cell 11 — Final Summary & Download

In [ ]:
import os, sys
REPO_ROOT = '/content/LLM-Assignment-3'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
from src.utils import save_json

save_json({
    'task_a': task_a_results, 'task_b': task_b_results,
    'ablations': ablation_results,
    'error_taxonomy': {'failure_distribution': taxonomy.get_failure_distribution(),
                       'failure_rates': taxonomy.get_failure_rate_by_category(),
                       'by_question_type': taxonomy.breakdown_by_metadata('question_type')},
    'diagnostic': {'coupling': coupling, 'synthesis': synthesis, 'coreference': coref},
    'faithfulness': {'aggregate': agg, 'refusal_rate': refusal_rate},
    'mode': 'CPU-only / BM25 / mock generation',
}, 'artifacts/results/all_results_a3.json')

best_a = task_a_results.get('bm25_full_history',list(task_a_results.values())[-1])['metrics']['overall']
best_b = task_b_results.get(best_gen_key,list(task_b_results.values())[-1])['metrics']['overall']
top_f  = list(taxonomy.get_failure_distribution().items())[0]

print('='*65)
print('  ASSIGNMENT 3 — FINAL RESULTS (CPU Mode, BM25)')
print('='*65)
print(f"TASK A  nDCG@1={best_a.get('ndcg@1',0):.4f}  nDCG@5={best_a.get('ndcg@5',0):.4f}  nDCG@10={best_a.get('ndcg@10',0):.4f}  MRR={best_a.get('mrr',0):.4f}")
print(f"TASK B  ROUGE-L={best_b.get('rouge_l_f1',0):.4f}  HM={best_b.get('harmonic_mean',0):.4f}  UnansF1={best_b.get('unanswerable_f1',0):.4f}")
print(f"FAITH   Mean={agg['mean_faithfulness']:.4f}  Hall={hall_rate:.4f}  Refusal={refusal_rate:.1%}")
print(f"DIAG    Ret-Gen r={coupling.get('pearson_correlation',0):.4f}  SynthGap={synthesis['gap']:.3f}  CorefGap={coref['gap']:.3f}")
print(f"        TopFail={top_f[0]} ({top_f[1]} cases)")
print('='*65)

In [ ]:
import subprocess
from google.colab import files
subprocess.run(['zip','-r','/content/A3_results.zip','artifacts/results/'], check=True)
files.download('/content/A3_results.zip')
print('Download started.')